Process Inputs: Chaining Prompts
Setup
Load the API key and relevant Python libaries.
In this course, we've provided some code that loads the OpenAI API key for you.

In [1]:
!pip install groq
from google.colab import userdata
from groq import Groq

client = Groq(
    api_key=userdata.get("GROQ_API_KEY")
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 2.9 MB/s eta 0:00:00


In [2]:
from openai import OpenAI
from google.colab import userdata

client = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

def get_completion_from_messages(
    messages,
    model="llama-3.3-70b-versatile",
    temperature=0,
    max_tokens=500
):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content


Check output for potentially harmful content

===

This code uses the OpenAI Moderation API to check whether the generated response is safe before sending it to the customer. It is commonly used in AI chatbots, customer support systems, and AI assistants to prevent harmful or policy-violating content.

In [7]:
from openai import OpenAI
from google.colab import userdata

client = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

text = """
The SmartX ProPhone has a 6.1-inch display...
"""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    temperature=0,
    messages=[
        {
            "role": "system",
            "content": """You are a content moderation assistant.

Return ONLY JSON in this format:

{
  "safe": true,
  "reason": "Safe product description"
}

If the text contains hate, violence, sexual content, self-harm, or illegal instructions,
return safe=false with the reason."""
        },
        {
            "role": "user",
            "content": text
        }
    ]
)

print(response.choices[0].message.content)

{
  "safe": true,
  "reason": "Safe product description"
}


Check if output is factually based on the provided product information

Configure Groq Client



In [8]:
from openai import OpenAI
from google.colab import userdata

client = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

Create the Completion Function

In [9]:
def get_completion_from_messages(
    messages,
    model="llama-3.3-70b-versatile",
    temperature=0,
    max_tokens=1
):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens
    )

    return response.choices[0].message.content.strip()

In [10]:
system_message = f"""
You are an assistant that evaluates whether
customer service agent responses sufficiently
answer customer questions, and also validates that
all the facts the assistant cites from the product
information are correct.

The product information and user and customer
service agent messages will be delimited by
3 backticks ```.

Respond with only one letter:

Y = The response correctly answers the customer's question
    AND all facts are correct.

N = Otherwise.

Output only Y or N.
"""

customer_message = """
tell me about the smartx pro phone and
the fotosnap camera, the dslr one.
Also tell me about your tvs
"""

product_information = """
{ "name": "SmartX ProPhone", "category": "Smartphones and Accessories", "brand": "SmartX", "model_number": "SX-PP10", "warranty": "1 year", "rating": 4.6, "features": ["6.1-inch display","128GB storage","12MP dual camera","5G"], "price": 899.99 }

{ "name": "FotoSnap DSLR Camera", "category": "Cameras and Camcorders", "brand": "FotoSnap", "model_number": "FS-DSLR200", "warranty": "1 year", "rating": 4.7, "features": ["24.2MP sensor","1080p video","3-inch LCD","Interchangeable lenses"], "price": 599.99 }

{ "name": "CineView 4K TV", "category": "Televisions and Home Theater Systems", "brand": "CineView", "model_number": "CV-4K55", "warranty": "2 years", "rating": 4.8, "features": ["55-inch display","4K resolution","HDR","Smart TV"], "price": 599.99 }

{ "name": "SoundMax Home Theater", "category": "Televisions and Home Theater Systems", "brand": "SoundMax", "model_number": "SM-HT100", "warranty": "1 year", "rating": 4.4, "features": ["5.1 channel","1000W output","Wireless subwoofer","Bluetooth"], "price": 399.99 }

{ "name": "CineView 8K TV", "category": "Televisions and Home Theater Systems", "brand": "CineView", "model_number": "CV-8K65", "warranty": "2 years", "rating": 4.9, "features": ["65-inch display","8K resolution","HDR","Smart TV"], "price": 2999.99 }

{ "name": "SoundMax Soundbar", "category": "Televisions and Home Theater Systems", "brand": "SoundMax", "model_number": "SM-SB50", "warranty": "1 year", "rating": 4.3, "features": ["2.1 channel","300W output","Wireless subwoofer","Bluetooth"], "price": 199.99 }

{ "name": "CineView OLED TV", "category": "Televisions and Home Theater Systems", "brand": "CineView", "model_number": "CV-OLED55", "warranty": "2 years", "rating": 4.7, "features": ["55-inch display","4K resolution","HDR","Smart TV"], "price": 1499.99 }
"""

q_a_pair = f"""
Customer message:
```{customer_message}```

Product information:
```{product_information}```

Agent response:
```{final_response_to_customer}```

Does the response correctly use the product information?

Does it sufficiently answer the customer's question?

Output only Y or N.
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": q_a_pair}
]

response = get_completion_from_messages(messages)

print(response)

Y


This code implements an AI-based Response Evaluation System. Instead of generating a response, it checks whether the customer service agent's response is accurate, complete, and consistent with the product information.

| Step | Code Section                     | Purpose                                                            |
| ---- | -------------------------------- | ------------------------------------------------------------------ |
| 1    | `system_message`                 | Defines the AI's role as a response evaluator.                     |
| 2    | `customer_message`               | Contains the customer's original question.                         |
| 3    | `product_information`            | Provides the product database used as the source of truth.         |
| 4    | `final_response_to_customer`     | Contains the customer service agent's reply.                       |
| 5    | `q_a_pair`                       | Combines the question, product data, and response into one prompt. |
| 6    | `messages`                       | Formats the prompt for the LLM.                                    |
| 7    | `get_completion_from_messages()` | Sends the prompt to the Groq model.                                |
| 8    | `print(response)`                | Prints either **Y** or **N**.                                      |


This code is an AI Response Validation System. It checks whether an agent's response is relevant, factually correct, and sufficiently answers the customer's question. In this example, the agent's response is "life is like a box of chocolates", which is unrelated to the customer's request, so the model should return N.

In [11]:
another_response = "life is like a box of chocolates"
q_a_pair = f"""
Customer message: ```{customer_message}```
Product information: ```{product_information}```
Agent response: ```{another_response}```

Does the response use the retrieved information correctly?
Does the response sufficiently answer the question?

Output Y or N
"""
messages = [
    {'role': 'system', 'content': system_message},
    {'role': 'user', 'content': q_a_pair}
]

response = get_completion_from_messages(messages)
print(response)

N


| Step | Code Section                     | Purpose                                                                                       |
| ---- | -------------------------------- | --------------------------------------------------------------------------------------------- |
| 1    | `another_response`               | Stores the agent's response to be evaluated.                                                  |
| 2    | `q_a_pair`                       | Combines the customer question, product information, and agent response into a single prompt. |
| 3    | `messages`                       | Formats the prompt for the Groq LLM (system + user messages).                                 |
| 4    | `get_completion_from_messages()` | Sends the prompt to the Groq model for evaluation.                                            |
| 5    | `print(response)`                | Displays the evaluation result (`Y` or `N`).                                                  |


| Step | Process                                                                           |
| ---- | --------------------------------------------------------------------------------- |
| 1    | Customer asks a question.                                                         |
| 2    | Agent or AI generates a response.                                                 |
| 3    | Product information is provided as the reference.                                 |
| 4    | The Groq model compares the response with the customer question and product data. |
| 5    | The model evaluates relevance, accuracy, and completeness.                        |
| 6    | Returns **`Y`** if valid, otherwise **`N`**.                                      |
